# Physical parameter candidate sets

This notebook assembles historical temperature and salinity results, extracts physical parameters and advection metadata, synchronizes completed runs with an Optuna study, and optionally requests new candidate parameter sets.

Its organization mirrors `CandidateParameterSets.ipynb`, but this workflow uses a different parameter list and treats the temperature/salinity advection configuration as a categorical choice. It intentionally stops after candidate generation. ROMS input-file generation will be designed separately later.

In [1]:
# Basic packages
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr
import pandas as pd
from IPython.display import display
import math
from netCDF4 import Dataset, num2date

# DateTime packages
from matplotlib.dates import DateFormatter
from datetime import datetime, timedelta
import time
import matplotlib.dates as mdates

# Stats packages
import scipy
import PyCO2SYS as pyco2
import gsw
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
import optuna

# Logistical packages
import requests
from importlib import reload
import warnings
import re
from pathlib import Path
import gc

# 1. Configuration

The switches below separate database updates from candidate generation. Both default to `False` so running the notebook for inspection does not complete trials or request new candidates accidentally.

`ACTIVE_NUMERIC_PARAMETERS` explicitly selects `nl_tnu2_tracer0` and `nl_tnu2_tracer1`. Parameters matching `EXCLUDED_NUMERIC_PARAMETER_PREFIXES`, along with tracer-indexed values outside `ACTIVE_TRACER_INDICES`, are retained in the extracted dataframe but excluded from Optuna. Replace `None` with an explicit list when the physical search space has been finalized.

In [2]:
# ============================================================
# PATHS
# ============================================================

COST_DIR = Path("/Users/akbaskind/Desktop/COST_FILES")
OPT_DIR = Path("/Users/akbaskind/Desktop/Optimization")
FILE_LIST = OPT_DIR / "FileNames.xlsx"
PARAMETER_LIST = OPT_DIR / "PhysicalParameterList.csv"
RUN_MAP_FILE = OPT_DIR / "runmap.xlsx"
DSTART_VARIABLE = "dstart"
DSTART_YEAR_COLUMN = "dstart_year"
HISTORY_ATTRIBUTE = "history"
RUN_DATE_COLUMN = "Run Date"
RUN_DATE_FALLBACK = pd.Timestamp("2000-01-01")

runs = pd.read_excel(FILE_LIST)
runs = runs[["Run Name", "Cost File", "Station File"]].dropna(
    subset=["Run Name", "Cost File", "Station File"]
)

parameter_names = (
    pd.read_csv(PARAMETER_LIST)["Variable"]
    .dropna()
    .astype(str)
    .str.strip()
    .tolist()
)

print(f"{len(runs)} runs")
print(f"{len(parameter_names)} base parameter names")

# ============================================================
# FILTERS AND THRESHOLDS
# ============================================================

COST_THRESHOLD = 200
MAX_MISMATCH_LOCATIONS = 10

# ============================================================
# METRIC DEFINITIONS FOR nRMSE AND COST
# ============================================================

metric_info = {
    "Surface Temperature": ("temp_mod", "temp_obs"),
    "Bottom Temperature": ("temp_mod", "temp_obs"),
    "Surface Salinity": ("salt_mod", "salt_obs"),
    "Bottom Salinity": ("salt_mod", "salt_obs"),
}

cost_components = {
    "Surface Temperature": "temperature_cost",
    "Bottom Temperature": "temperature_cost",
    "Surface Salinity": "salt_cost",
    "Bottom Salinity": "salt_cost",
}

cost_weights = {target: 1 for target in cost_components}

print(f"\nNumber of nRMSE metrics: {len(metric_info)}")
print(f"Metrics: {list(metric_info)}")

# ============================================================
# CATEGORICAL ADVECTION SEARCH SPACE
# ============================================================

ADVECTION_ATTRIBUTE = "NLM_TADV"
ADVECTION_COLUMNS = [
    "temp_horizontal_advection",
    "temp_vertical_advection",
    "salt_horizontal_advection",
    "salt_vertical_advection",
]

# Each label describes one complete, internally consistent four-field bundle.
# Remove a bundle here if it should not be available to future Optuna trials.
ADVECTION_SCHEMES = {
    "Upstream3_Centered4": {
        "temp_horizontal_advection": "Upstream3",
        "temp_vertical_advection": "Centered4",
        "salt_horizontal_advection": "Upstream3",
        "salt_vertical_advection": "Centered4",
    },
    "HSIMT": {
        column: "HSIMT" for column in ADVECTION_COLUMNS
    },
    "Akima4": {
        column: "Akima4" for column in ADVECTION_COLUMNS
    },
    "MPDATA": {
        column: "MPDATA" for column in ADVECTION_COLUMNS
    },
}

CATEGORICAL_PARAMETER = "advection_scheme"
ACTIVE_CATEGORICAL_PARAMETERS = [CATEGORICAL_PARAMETER]

# Prefix-based exclusions are applied even when ACTIVE_NUMERIC_PARAMETERS is None.
# The extracted columns remain in DF_opt for inspection and future reconsideration.
EXCLUDED_NUMERIC_PARAMETER_PREFIXES = ("Tobc_",)

# Only temperature (tracer0) and salinity (tracer1) are relevant here.
# Other tracer-indexed columns remain available in DF_opt for the audit below.
ACTIVE_TRACER_INDICES = (0, 1)

# Keep the factorial search space explicit so newly varied archive columns cannot
# enter the optimization accidentally.
ACTIVE_NUMERIC_PARAMETERS = [
    "nl_tnu2_tracer0",
    "nl_tnu2_tracer1",
]

# Controlled 3 x 4 factorial grid. Both tracer parameters receive the same value.
NL_TNU2_GRID = (0.5, 1.0, 2.0)
GRID_ADVECTION_SCHEMES = tuple(ADVECTION_SCHEMES)
FACTORIAL_GRID_VERSION = "physical_tied_grid_v1"

# ============================================================
# OPTUNA STUDY SETTINGS
# ============================================================

STUDY_NAME = "model_calibration_physical"
CURRENT_STUDY_NAME = STUDY_NAME
STORAGE_FILE = OPT_DIR / "model_calibration_physical.db"
STORAGE = f"sqlite:///{STORAGE_FILE}"
OBJECTIVE_VERSION = "cost_v2_physical"

# Load and validate the shared mapping from study-assigned names to model run names.
required_run_map_columns = [
    "Actual Run Name", "Study Name", "Study Run Name", "Trial Number"
]
run_map = pd.read_excel(RUN_MAP_FILE)
run_map.columns = run_map.columns.astype(str).str.strip()
missing_run_map_columns = [
    column for column in required_run_map_columns if column not in run_map.columns
]
if missing_run_map_columns:
    raise ValueError(
        f"Run map is missing required columns: {missing_run_map_columns}"
    )

for column in ["Actual Run Name", "Study Name", "Study Run Name"]:
    run_map[column] = run_map[column].astype("string").str.strip()

trial_numbers = pd.to_numeric(run_map["Trial Number"], errors="coerce")
invalid_trial_numbers = trial_numbers.isna() | (trial_numbers % 1 != 0)
if invalid_trial_numbers.any():
    invalid_rows = (run_map.index[invalid_trial_numbers] + 2).tolist()
    raise ValueError(
        f"Run map has invalid Trial Number values in Excel rows: {invalid_rows}"
    )
run_map["Trial Number"] = trial_numbers.astype(int)

current_study_run_map = run_map.loc[
    run_map["Study Name"].eq(CURRENT_STUDY_NAME)
].copy()
duplicate_mapping_keys = [
    ["Study Name", "Trial Number"],
    ["Study Name", "Study Run Name"],
    ["Study Name", "Actual Run Name"],
]
duplicate_mapping_messages = []
for columns in duplicate_mapping_keys:
    duplicate_rows = current_study_run_map.loc[
        current_study_run_map.duplicated(columns, keep=False), columns
    ]
    if not duplicate_rows.empty:
        duplicate_mapping_messages.append(
            f"{columns}: {duplicate_rows.to_dict('records')}"
        )
if duplicate_mapping_messages:
    raise ValueError(
        f"Invalid duplicate run-map mappings for {CURRENT_STUDY_NAME!r}:\n"
        + "\n".join(duplicate_mapping_messages)
    )

UPDATE_OPTUNA_WITH_COMPLETED_RUNS = False
GENERATE_NEW_CANDIDATES = True

print(f"\nStudy name: {STUDY_NAME}")
print(f"Storage:    {STORAGE_FILE}")
print(f"Run-map rows for this study: {len(current_study_run_map)}")

110 runs
55 base parameter names

Number of nRMSE metrics: 4
Metrics: ['Surface Temperature', 'Bottom Temperature', 'Surface Salinity', 'Bottom Salinity']

Study name: model_calibration_physical
Storage:    /Users/akbaskind/Desktop/Optimization/model_calibration_physical.db
Run-map rows for this study: 12


# 2. Functions

## 2.1. nRMSE functions

The signed normalized RMSD calculation is unchanged from `CandidateParameterSets.ipynb`. Model and observation arrays must have matching dimensions, and statistics use paired finite values only. Data-quality problems are retained in `DF_rmse_diagnostics`.

In [3]:
# ============================================================
# OBSERVATIONAL ERROR
# ============================================================

def get_stderr(var_obs, obs_mean):

    if var_obs.startswith("sed"):
        return obs_mean

    elif var_obs.startswith("pH"):
        return 0.1

    elif var_obs.startswith("temp"):
        return 0.01

    elif var_obs.startswith("salt"):
        return 0.005 * obs_mean

    elif var_obs.startswith("oxy"):
        return 3.125

    elif var_obs.startswith("N"):
        return 0.1

    elif var_obs.startswith("Si"):
        return 0.1

    elif var_obs.startswith("SD"):
        return 0.2

    else:
        return 0


# ============================================================
# CALCULATE ONE RMSE*
# ============================================================

def describe_locations(data_array, mask, limit):
    """Return readable coordinate labels for True entries in a mask."""

    locations = []

    for index_values in np.argwhere(mask)[:limit]:
        labels = []

        for dim, index in zip(data_array.dims, index_values):
            if dim in data_array.coords and data_array.coords[dim].ndim == 1:
                coordinate = data_array.coords[dim].values[index]
                labels.append(f"{dim}={coordinate}")
            else:
                labels.append(f"{dim}[{index}]")

        locations.append(", ".join(labels) if labels else "scalar")

    return locations


def record_rmse_issue(
    diagnostics,
    run_name,
    metric_name,
    issue,
    details,
    print_details=True,
):
    """Print and retain one RMSE data-quality issue."""

    record = {
        "Run Name": run_name,
        "Metric": metric_name,
        "Issue": issue,
        "Details": details,
    }
    diagnostics.append(record)
    if print_details:
        print(f"  RMSE WARNING [{metric_name}] {issue}: {details}")


def calculate_rmse_star(
    ds,
    metric_name,
    var_mod,
    var_obs,
    run_name,
    diagnostics,
    max_mismatch_locations=10,
):
    """Calculate signed RMSD*' from paired, finite values."""

    missing_variables = [
        variable
        for variable in (var_mod, var_obs)
        if variable not in ds.variables
    ]

    if missing_variables:
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Missing variable",
            f"not found: {missing_variables}",
        )
        return np.nan

    mod_data = ds[var_mod]
    obs_data = ds[var_obs]

    if metric_name.startswith("Surface"):
        mod_data = mod_data.isel(Depth=0)
        obs_data = obs_data.isel(Depth=0)
    elif metric_name.startswith("Bottom"):
        mod_data = mod_data.isel(Depth=1)
        obs_data = obs_data.isel(Depth=1)

    # Pairing is meaningful only when dimensions and shapes agree exactly.
    if mod_data.dims != obs_data.dims or mod_data.shape != obs_data.shape:
        details = (
            f"model dims/shape={mod_data.dims}/{mod_data.shape}; "
            f"observation dims/shape={obs_data.dims}/{obs_data.shape}"
        )
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Alignment mismatch", details
        )
        return np.nan

    mod_values = np.asarray(mod_data.values)
    obs_values = np.asarray(obs_data.values)
    mod_finite = np.isfinite(mod_values)
    obs_finite = np.isfinite(obs_values)

    model_only = mod_finite & ~obs_finite
    observation_only = ~mod_finite & obs_finite

    if model_only.any() or observation_only.any():
        model_locations = describe_locations(
            mod_data, model_only, max_mismatch_locations
        )
        observation_locations = describe_locations(
            obs_data, observation_only, max_mismatch_locations
        )
        details = (
            f"model finite/observation non-finite={model_only.sum()} "
            f"at {model_locations}; "
            f"observation finite/model non-finite={observation_only.sum()} "
            f"at {observation_locations}"
        )
        record_rmse_issue(
            diagnostics,
            run_name,
            metric_name,
            "Missing-data mismatch",
            details,
            print_details=False,
        )
        print(
            f"  RMSE WARNING [{metric_name}] missing-data mismatch: "
            f"model-only={model_only.sum()}, "
            f"observation-only={observation_only.sum()}"
        )

    # Use the same paired sample for every statistic.
    paired = mod_finite & obs_finite
    mod = mod_values[paired].astype(float)
    obs = obs_values[paired].astype(float)

    if mod.size < 2:
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Insufficient paired data",
            f"found {mod.size} paired finite value(s); at least 2 are required",
        )
        return np.nan

    mod_mean = mod.mean()
    obs_mean = obs.mean()
    std_mod = mod.std()
    std_obs_sample = obs.std()

    # Include the configured observational uncertainty in observation spread.
    stderr = get_stderr(var_obs, obs_mean)
    std_obs = np.sqrt(std_obs_sample**2 + stderr**2)

    if std_mod == 0 or not np.isfinite(std_mod):
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Invalid model variance",
            f"model standard deviation={std_mod}",
        )
        return np.nan

    if std_obs == 0 or not np.isfinite(std_obs):
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Invalid observation variance",
            f"adjusted observation standard deviation={std_obs}",
        )
        return np.nan

    covariance = np.mean((mod - mod_mean) * (obs - obs_mean))
    correlation = covariance / (std_mod * std_obs)
    std_norm = std_mod / std_obs
    sign = np.sign(std_mod - std_obs)

    inside = 1 + std_norm**2 - 2 * std_norm * correlation

    # Roundoff can produce a tiny negative value; a larger one is not valid.
    if inside < -1e-12 or not np.isfinite(inside):
        record_rmse_issue(
            diagnostics, run_name, metric_name, "Invalid RMSD expression",
            f"value under square root={inside}",
        )
        return np.nan

    rmsd_star = np.sqrt(max(inside, 0.0)) * sign
    return float(rmsd_star)

## 2.2. Cost function

The total physical cost is the weighted sum of the configured surface and bottom temperature and salinity cost components. Lower values are better.

In [4]:
def calculate_cost(ds, cost_components, cost_weights):
    """Sum weighted surface, bottom, and non-depth-specific costs."""

    total_cost = 0.0

    for target, variable_name in cost_components.items():
        weight = cost_weights[target]

        if target.startswith("Surface"):
            component = float(ds[variable_name][0])
        elif target.startswith("Bottom"):
            component = float(ds[variable_name][1])
        else:
            component = float(ds[variable_name])

        total_cost += component * weight**2

    return total_cost

## 2.3. Numeric parameter extraction

Scalar NetCDF variables remain one column. Arrays are expanded into one column per element so each value can be inspected and, if selected, optimized separately. The `dstart` time variable is decoded separately and retained as the numeric parameter `dstart_year`.

In [5]:
def get_parameter_values(ds, parameter_name):

    # Parameter doesn't exist in this file
    # Don't create a new scalar-named column
    if parameter_name not in ds.variables:
        return {}

    var = ds.variables[parameter_name]

    try:

        # --------------------------------
        # Scalar parameter
        # --------------------------------
        if var.ndim == 0:

            value = var[...]

            if np.ma.is_masked(value):
                value = np.nan

            return {
                parameter_name: np.asarray(value).item()
            }

        # --------------------------------
        # Parameter with dimensions
        # --------------------------------
        values = np.asarray(var[:]).squeeze()

        # Squeezes down to a scalar
        if values.ndim == 0:

            return {
                parameter_name: values.item()
            }

        # --------------------------------
        # One-dimensional parameter
        # --------------------------------
        if values.ndim == 1:

            dim_name = var.dimensions[0]

            return {
                f"{parameter_name}_{dim_name}{i}": value
                for i, value in enumerate(values)
            }

        # --------------------------------
        # Anything else
        # --------------------------------
        values = values.ravel()

        return {
            f"{parameter_name}_{i}": value
            for i, value in enumerate(values)
        }

    except Exception as e:

        print(
            f"    Could not read {parameter_name}: {e}"
        )

        return {}


def get_dstart_year(ds, variable_name=DSTART_VARIABLE):
    """Decode the station-file start date and return only its year."""

    if variable_name not in ds.variables:
        return np.nan

    var = ds.variables[variable_name]
    raw_value = var[...]

    if np.ma.is_masked(raw_value):
        return np.nan

    value = np.asarray(raw_value).squeeze()
    if value.size != 1:
        return np.nan

    if np.issubdtype(value.dtype, np.datetime64):
        timestamp = pd.to_datetime(value.reshape(1))[0]
    else:
        units = getattr(var, "units", None)
        if units is None:
            raise ValueError(f"{variable_name} has no NetCDF time units")
        timestamp = num2date(
            value.item(),
            units=units,
            calendar=getattr(var, "calendar", "standard"),
            only_use_cftime_datetimes=False,
        )

    return int(timestamp.year)


def get_run_date(ds, attribute_name=HISTORY_ATTRIBUTE):
    """Extract the model run date from the station-file history attribute."""

    # ROMS history strings contain a date such as 'January 18, 2026'.
    # Return the documented fallback date when the attribute is absent or malformed.
    history = getattr(ds, attribute_name, None)
    if history is None:
        return RUN_DATE_FALLBACK, True

    date_match = re.search(
        r"\b(?:January|February|March|April|May|June|July|August|"
        r"September|October|November|December)\s+\d{1,2},\s+\d{4}\b",
        str(history),
    )
    if date_match is None:
        return RUN_DATE_FALLBACK, True

    run_date = pd.to_datetime(
        date_match.group(0),
        format="%B %d, %Y",
        errors="coerce",
    )
    if pd.isna(run_date):
        return RUN_DATE_FALLBACK, True

    return run_date.normalize(), False

## 2.4. Categorical advection extraction

`NLM_TADV` is a multiline global NetCDF attribute. The parser retains the four raw fields for auditing. A second function maps an exact four-field combination to one configured bundle label. Unrecognized or incomplete combinations are not silently added to the search space.

In [6]:
def get_temp_salt_advection(ds, attribute_name = "NLM_TADV"):
    """
    Extract horizontal and vertical advection schemes for temp and salt
    from a multiline NetCDF global attribute.
    """

    output = {
        "temp_horizontal_advection": np.nan,
        "temp_vertical_advection": np.nan,
        "salt_horizontal_advection": np.nan,
        "salt_vertical_advection": np.nan,
    }

    # Return NaNs if the global attribute does not exist
    if attribute_name not in ds.ncattrs():
        return output

    advection_text = ds.getncattr(attribute_name)

    for variable in ("temp", "salt"):

        match = re.search(
            rf"^\s*{re.escape(variable)}:\s+(\S+)\s+(\S+)",
            str(advection_text),
            flags=re.MULTILINE,
        )

        if match:
            output[f"{variable}_horizontal_advection"] = match.group(1)
            output[f"{variable}_vertical_advection"] = match.group(2)

    return output


def identify_advection_scheme(advection_values, schemes=ADVECTION_SCHEMES):
    """Return the label matching all four extracted advection fields."""

    if any(pd.isna(advection_values.get(column)) for column in ADVECTION_COLUMNS):
        return np.nan

    for label, expected_values in schemes.items():
        if all(
            advection_values[column] == expected_values[column]
            for column in ADVECTION_COLUMNS
        ):
            return label

    return np.nan

# 3. Get nRMSE, cost, numeric parameters, and advection categories

Each listed run is processed once. Cost files supply the objectives; station files supply physical parameter values, the decoded `dstart_year`, and the global advection attribute. Everything is assembled into one row per run.

In [7]:
all_results = []
rmse_diagnostics = []
missing_cost_files = []
missing_station_files = []
fallback_run_date_runs = []

for run_number, (_, row) in enumerate(runs.iterrows(), start=1):
    run_name = row["Run Name"]
    cost_file = row["Cost File"]
    station_file = row["Station File"]
    cost_path = COST_DIR / cost_file
    station_path = COST_DIR / station_file

    print(f"\n[{run_number}/{len(runs)}] Processing {run_name}")

    # All outputs for this run accumulate in one dictionary and become one row.
    run_results = {
        "Run Name": run_name,
        "Cost File": cost_file,
        "Station File": station_file,
    }

    try:
        with xr.open_dataset(cost_path) as ds:
            for metric_name, (var_mod, var_obs) in metric_info.items():
                run_results[metric_name] = calculate_rmse_star(
                    ds, metric_name, var_mod, var_obs, run_name,
                    rmse_diagnostics, MAX_MISMATCH_LOCATIONS,
                )
            run_results["Cost"] = calculate_cost(
                ds, cost_components, cost_weights
            )
        run_results["Cost Status"] = "Success"
    except Exception as error:
        print(f"  COST FILE ERROR: {error}")
        run_results["Cost Status"] = f"ERROR: {error}"
        for metric_name in metric_info:
            run_results.setdefault(metric_name, np.nan)
        run_results.setdefault("Cost", np.nan)
        missing_cost_files.append(cost_path.name)

    try:
        with Dataset(station_path, mode="r") as ds:
            for parameter in parameter_names:
                run_results.update(get_parameter_values(ds, parameter))

            run_results[DSTART_YEAR_COLUMN] = get_dstart_year(ds)

            # Preserve the run date recorded in this station file's history.
            run_date, used_run_date_fallback = get_run_date(ds)
            run_results[RUN_DATE_COLUMN] = run_date
            if used_run_date_fallback:
                fallback_run_date_runs.append(run_name)

            advection_values = get_temp_salt_advection(
                ds, attribute_name=ADVECTION_ATTRIBUTE
            )
            run_results.update(advection_values)
            run_results[CATEGORICAL_PARAMETER] = identify_advection_scheme(
                advection_values
            )

        run_results["Parameter Status"] = "Success"
    except Exception as error:
        print(f"  STATION FILE ERROR: {error}")
        run_results["Parameter Status"] = f"ERROR: {error}"
        for parameter in parameter_names:
            run_results.setdefault(parameter, np.nan)
        run_results.setdefault(DSTART_YEAR_COLUMN, np.nan)
        run_results.setdefault(RUN_DATE_COLUMN, RUN_DATE_FALLBACK)
        if run_name not in fallback_run_date_runs:
            fallback_run_date_runs.append(run_name)
        for column in [*ADVECTION_COLUMNS, CATEGORICAL_PARAMETER]:
            run_results.setdefault(column, np.nan)
        missing_station_files.append(station_path.name)

    all_results.append(run_results)
    gc.collect()

DF_opt = pd.DataFrame(all_results).dropna(axis=1, how="all")

if fallback_run_date_runs:
    print(
        f"Runs assigned fallback date {RUN_DATE_FALLBACK.date()}: "
        f"{fallback_run_date_runs}"
    )


[1/110] Processing A13
  RMSE WARNING [Surface Temperature] missing-data mismatch: model-only=1064, observation-only=0
  RMSE WARNING [Bottom Temperature] missing-data mismatch: model-only=1472, observation-only=0
  RMSE WARNING [Surface Salinity] missing-data mismatch: model-only=1064, observation-only=0
  RMSE WARNING [Bottom Salinity] missing-data mismatch: model-only=1526, observation-only=0

[2/110] Processing DU_Mar16
  RMSE WARNING [Surface Temperature] missing-data mismatch: model-only=28, observation-only=0
  RMSE WARNING [Bottom Temperature] missing-data mismatch: model-only=42, observation-only=0
  RMSE WARNING [Surface Salinity] missing-data mismatch: model-only=28, observation-only=0
  RMSE WARNING [Bottom Salinity] missing-data mismatch: model-only=42, observation-only=0

[3/110] Processing DU
  RMSE WARNING [Surface Temperature] missing-data mismatch: model-only=1064, observation-only=0
  RMSE WARNING [Bottom Temperature] missing-data mismatch: model-only=1472, observat

## 3.1. Data-loading and category diagnostics

The tables below identify failed files, large costs, RMSE data-quality issues, missing advection metadata, and the observed frequency of each complete advection bundle. A complete but unrecognized combination indicates that `ADVECTION_SCHEMES` needs a deliberate update.

In [8]:
failed_cost_runs = DF_opt.loc[
    DF_opt["Cost Status"].ne("Success"), "Run Name"
].tolist()
failed_parameter_runs = DF_opt.loc[
    DF_opt["Parameter Status"].ne("Success"), "Run Name"
].tolist()
high_cost_runs = DF_opt.loc[
    DF_opt["Cost"].ge(COST_THRESHOLD), "Run Name"
].tolist()

print(f"Runs with cost-file errors: {failed_cost_runs}")
print(f"Runs with station-file errors: {failed_parameter_runs}")
print(f"Runs at or above the cost threshold: {high_cost_runs}")

DF_rmse_diagnostics = pd.DataFrame(
    rmse_diagnostics,
    columns=["Run Name", "Metric", "Issue", "Details"],
)
print(f"RMSE diagnostic records: {len(DF_rmse_diagnostics)}")
if not DF_rmse_diagnostics.empty:
    display(DF_rmse_diagnostics)

advection_complete = DF_opt[ADVECTION_COLUMNS].notna().all(axis=1)
unrecognized_advection = DF_opt.loc[
    advection_complete & DF_opt[CATEGORICAL_PARAMETER].isna(),
    ["Run Name", *ADVECTION_COLUMNS],
]

print("\nObserved advection bundles:")
display(
    DF_opt.groupby([CATEGORICAL_PARAMETER, *ADVECTION_COLUMNS], dropna=False)
    .size()
    .rename("Runs")
    .sort_values(ascending=False)
    .reset_index()
)

if not unrecognized_advection.empty:
    print("Complete advection combinations not present in ADVECTION_SCHEMES:")
    display(unrecognized_advection)

Runs with cost-file errors: ['2005_CTRL', 'C4', 'A3', 'C13']
Runs with station-file errors: ['C13']
Runs at or above the cost threshold: []
RMSE diagnostic records: 424


,Run Name,Metric,Issue,Details
0,A13,Surface Temperature,Missing-data mismatch,model finite/observation non-finite=1064 at ['...
1,A13,Bottom Temperature,Missing-data mismatch,model finite/observation non-finite=1472 at ['...
2,A13,Surface Salinity,Missing-data mismatch,model finite/observation non-finite=1064 at ['...
3,A13,Bottom Salinity,Missing-data mismatch,model finite/observation non-finite=1526 at ['...
4,DU_Mar16,Surface Temperature,Missing-data mismatch,model finite/observation non-finite=28 at ['Si...
...,...,...,...,...
419,OPTUNA_66,Bottom Salinity,Missing-data mismatch,model finite/observation non-finite=1526 at ['...
420,OPTUNA_67,Surface Temperature,Missing-data mismatch,model finite/observation non-finite=1064 at ['...
421,OPTUNA_67,Bottom Temperature,Missing-data mismatch,model finite/observation non-finite=1472 at ['...
422,OPTUNA_67,Surface Salinity,Missing-data mismatch,model finite/observation non-finite=1064 at ['...



Observed advection bundles:


,advection_scheme,temp_horizontal_advection,temp_vertical_advection,salt_horizontal_advection,salt_vertical_advection,Runs
0,Upstream3_Centered4,Upstream3,Centered4,Upstream3,Centered4,42
1,HSIMT,HSIMT,HSIMT,HSIMT,HSIMT,37
2,Akima4,Akima4,Akima4,Akima4,Akima4,28
3,NaN,NaN,NaN,NaN,NaN,2
4,MPDATA,MPDATA,MPDATA,MPDATA,MPDATA,1


## 3.2. Tracer-dimension audit

Parameters stored over the NetCDF `tracer` dimension are expanded into columns ending in `_tracer#`. This audit checks whether the available tracer values are identical within each run and, separately, whether `tracer0` and `tracer1` match.

The audit describes historical behavior only. Equal historical values mean the archive cannot distinguish their separate effects; it does not require future temperature and salinity values to remain tied.

In [9]:
# Group expanded columns by the parameter name before the _tracer# suffix.
tracer_column_groups = {}
for column in DF_opt.columns:
    match = re.match(r"^(?P<parameter>.+)_tracer(?P<index>\d+)$", column)
    if match:
        tracer_column_groups.setdefault(match.group("parameter"), []).append(
            (int(match.group("index")), column)
        )

tracer_audit_rows = []
for parameter, indexed_columns in sorted(tracer_column_groups.items()):
    indexed_columns = sorted(indexed_columns)
    tracer_columns = [column for _, column in indexed_columns]
    values = DF_opt[tracer_columns].apply(pd.to_numeric, errors="coerce")

    # Count distinct non-missing values across every available tracer in each run.
    has_tracer_values = values.notna().any(axis=1)
    all_available_equal = values.nunique(axis=1, dropna=True).le(1) & has_tracer_values

    tracer0_column = f"{parameter}_tracer0"
    tracer1_column = f"{parameter}_tracer1"
    first_two_available = (
        values[[tracer0_column, tracer1_column]].notna().all(axis=1)
        if tracer0_column in values and tracer1_column in values
        else pd.Series(False, index=values.index)
    )
    first_two_equal = (
        values[tracer0_column].eq(values[tracer1_column]) & first_two_available
        if tracer0_column in values and tracer1_column in values
        else pd.Series(False, index=values.index)
    )

    compared_all = int(has_tracer_values.sum())
    compared_first_two = int(first_two_available.sum())
    tracer_audit_rows.append({
        "Parameter": parameter,
        "Tracer Columns Found": len(tracer_columns),
        "Runs Compared Across Tracers": compared_all,
        "All Available Tracers Equal": int(all_available_equal.sum()),
        "All Available Equal Fraction": (
            all_available_equal.sum() / compared_all if compared_all else np.nan
        ),
        "Runs Compared tracer0/tracer1": compared_first_two,
        "tracer0 Equals tracer1": int(first_two_equal.sum()),
        "tracer0/tracer1 Equal Fraction": (
            first_two_equal.sum() / compared_first_two if compared_first_two else np.nan
        ),
    })

DF_tracer_audit = pd.DataFrame(tracer_audit_rows)
display(DF_tracer_audit)

,Parameter,Tracer Columns Found,Runs Compared Across Tracers,All Available Tracers Equal,All Available Equal Fraction,Runs Compared tracer0/tracer1,tracer0 Equals tracer1,tracer0/tracer1 Equal Fraction
0,Akt_bak,19,109,66,0.605505,109,109,1.0
1,LnudgeTCLM,19,109,109,1.000000,109,109,1.0
2,LtracerCLM,19,109,109,1.000000,109,109,1.0
3,LtracerSponge,19,109,109,1.000000,109,109,1.0
4,LtracerSrc,19,109,109,1.000000,109,109,1.0
5,Tnudg,19,109,109,1.000000,109,109,1.0
6,nl_tnu2,19,109,100,0.917431,109,109,1.0


# 4. Build the optimization dataframe

## 4.1. Objective nRMSE

The signed nRMSE values are converted to absolute values so that Optuna can minimize distance from zero regardless of the original sign.

In [10]:
active_rmse_cols = list(metric_info)
active_objective_cols = []

for column in active_rmse_cols:
    objective_column = f"{column} Objective"
    DF_opt[objective_column] = DF_opt[column].abs()
    active_objective_cols.append(objective_column)

display(
    DF_opt[active_objective_cols]
    .describe().T[["min", "25%", "50%", "75%", "max"]]
)

,min,25%,50%,75%,max
Surface Temperature Objective,0.084188,0.084511,0.086146,0.163168,0.249661
Bottom Temperature Objective,0.086345,0.087689,0.096880,0.270127,0.376286
Surface Salinity Objective,0.275946,0.401605,0.415503,0.950760,4.971296
Bottom Salinity Objective,0.444924,0.463922,0.514770,0.943859,1.433228


## 4.2. Keep runs with usable costs

Runs with missing or non-finite costs, along with runs at or above `COST_THRESHOLD`, are excluded from optimization.

In [11]:
old_count = len(DF_opt)
usable_cost = np.isfinite(DF_opt["Cost"]) & DF_opt["Cost"].lt(COST_THRESHOLD)
DF_opt = DF_opt.loc[usable_cost].reset_index(drop=True)

print(f"Number of usable runs: {len(DF_opt)}")
print(f"Number of runs dropped: {old_count - len(DF_opt)}")

Number of usable runs: 106
Number of runs dropped: 4


## 4.3. Separate numeric and categorical parameters

Numeric parameters receive continuous `FloatDistribution` bounds. The advection bundle receives a `CategoricalDistribution`. Keeping these paths separate prevents string categories from entering min/max calculations or `suggest_float`.

In [12]:
metadata_columns = {
    "Run Name", "Cost File", "Station File", "Cost",
    "Cost Status", "Parameter Status", RUN_DATE_COLUMN,
    *active_rmse_cols, *active_objective_cols,
    *ADVECTION_COLUMNS, CATEGORICAL_PARAMETER,
}

numeric_parameter_cols = [
    column for column in DF_opt.select_dtypes(include="number").columns
    if column not in metadata_columns
]
numeric_variation = DF_opt[numeric_parameter_cols].nunique(dropna=True).sort_values()
varied_numeric_parameter_cols = numeric_variation[numeric_variation > 1].index.tolist()

# Exclude whole parameter families by prefix.
excluded_by_prefix = [
    parameter
    for parameter in varied_numeric_parameter_cols
    if parameter.startswith(EXCLUDED_NUMERIC_PARAMETER_PREFIXES)
]

# Also exclude expanded tracer values other than tracer0 and tracer1.
excluded_by_tracer_index = []
for parameter in varied_numeric_parameter_cols:
    tracer_match = re.search(r"_tracer(?P<index>\d+)$", parameter)
    if tracer_match and int(tracer_match.group("index")) not in ACTIVE_TRACER_INDICES:
        excluded_by_tracer_index.append(parameter)

# Preserve parameter order while removing any duplicate exclusion reasons.
excluded_numeric_parameters = list(dict.fromkeys(
    excluded_by_prefix + excluded_by_tracer_index
))
eligible_numeric_parameters = [
    parameter
    for parameter in varied_numeric_parameter_cols
    if parameter not in excluded_numeric_parameters
]

if ACTIVE_NUMERIC_PARAMETERS is None:
    active_numeric_parameters = eligible_numeric_parameters.copy()
else:
    active_numeric_parameters = list(ACTIVE_NUMERIC_PARAMETERS)

missing_active_numeric = sorted(
    set(active_numeric_parameters).difference(eligible_numeric_parameters)
)
if missing_active_numeric:
    raise ValueError(
        "Active numeric parameters are unavailable, constant, or excluded: "
        f"{missing_active_numeric}"
    )

active_parameters = active_numeric_parameters + ACTIVE_CATEGORICAL_PARAMETERS

print(f"Varied numeric parameters: {len(varied_numeric_parameter_cols)}")
print(f"Excluded by prefix: {len(excluded_by_prefix)}")
print(excluded_by_prefix)
print(f"Excluded by tracer index: {len(excluded_by_tracer_index)}")
print(excluded_by_tracer_index)
print(f"Active numeric parameters: {len(active_numeric_parameters)}")
print(f"Active categorical parameters: {ACTIVE_CATEGORICAL_PARAMETERS}")
print(active_numeric_parameters)

Varied numeric parameters: 72
Excluded by prefix: 40
['Tobc_in_23', 'Tobc_in_22', 'Tobc_in_18', 'Tobc_in_17', 'Tobc_out_54', 'Tobc_out_53', 'Tobc_out_52', 'Tobc_out_50', 'Tobc_out_48', 'Tobc_out_45', 'Tobc_out_42', 'Tobc_out_41', 'Tobc_out_40', 'Tobc_out_39', 'Tobc_out_37', 'Tobc_out_36', 'Tobc_out_32', 'Tobc_out_31', 'Tobc_out_30', 'Tobc_out_28', 'Tobc_out_23', 'Tobc_out_22', 'Tobc_out_18', 'Tobc_out_17', 'Tobc_in_54', 'Tobc_in_53', 'Tobc_in_52', 'Tobc_in_50', 'Tobc_in_48', 'Tobc_in_45', 'Tobc_in_42', 'Tobc_in_41', 'Tobc_in_40', 'Tobc_in_39', 'Tobc_in_37', 'Tobc_in_36', 'Tobc_in_32', 'Tobc_in_31', 'Tobc_in_30', 'Tobc_in_28']
Excluded by tracer index: 30
['Akt_bak_tracer16', 'Akt_bak_tracer15', 'Akt_bak_tracer14', 'Akt_bak_tracer13', 'Akt_bak_tracer12', 'Akt_bak_tracer11', 'Akt_bak_tracer10', 'Akt_bak_tracer9', 'Akt_bak_tracer8', 'Akt_bak_tracer7', 'Akt_bak_tracer6', 'Akt_bak_tracer5', 'Akt_bak_tracer4', 'Akt_bak_tracer3', 'Akt_bak_tracer2', 'nl_tnu2_tracer11', 'nl_tnu2_tracer10', 'nl_

## 4.4. Build the Optuna dataframe

The optimization table retains run identity, total cost, active parameters, and individual objective components. Rows missing any active parameter remain visible here but are skipped during historical-trial import.

In [13]:
DF_optuna = DF_opt[
    ["Run Name", RUN_DATE_COLUMN, "Cost", *active_parameters, *active_objective_cols]
].copy()

print(DF_optuna.shape)
display(DF_optuna.head())

(106, 10)


,Run Name,Run Date,Cost,nl_tnu2_tracer0,nl_tnu2_tracer1,advection_scheme,Surface Temperature Objective,Bottom Temperature Objective,Surface Salinity Objective,Bottom Salinity Objective
0,A13,2026-01-18,3.137656,1.0,1.0,HSIMT,0.181582,0.270513,0.950856,0.944296
1,DU_Mar16,2026-03-12,37.189042,0.5,0.5,MPDATA,0.249661,0.376286,4.971296,1.414186
2,DU,2026-03-07,1.812291,0.5,0.5,Akima4,0.119669,0.097866,0.710441,0.770320
3,A17,2026-01-22,0.526578,1.0,1.0,HSIMT,0.086146,0.096880,0.394081,0.514770
4,C11,2026-04-23,0.592719,0.5,0.5,Upstream3_Centered4,0.085403,0.089563,0.425204,0.476297


# 5. Load the Optuna study

## 5.1. Create or load the study

The physical study uses a separate SQLite database and objective-version label. Do not combine trials calculated with a different cost definition in this study.

In [14]:
sampler = optuna.samplers.TPESampler(
    n_startup_trials=10,
    multivariate=True,
    seed=42,
)

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    direction="minimize",
    sampler=sampler,
    load_if_exists=True,
)

stored_objective_version = study.user_attrs.get("Objective Version")
if stored_objective_version is None:
    study.set_user_attr("Objective Version", OBJECTIVE_VERSION)
elif stored_objective_version != OBJECTIVE_VERSION:
    raise ValueError(
        "Objective-version mismatch:\n"
        f"  Study:    {stored_objective_version}\n"
        f"  Notebook: {OBJECTIVE_VERSION}"
    )

print(f"Loaded study: {study.study_name}")
print(f"Objective version: {OBJECTIVE_VERSION}")
print(f"Stored trials: {len(study.trials)}")

/var/folders/9k/5r38tm8d21g6nm3w9rchrd6m0000gn/T/ipykernel_87125/2877051632.py:1: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
[I 2026-09-16 13:31:43,706] A new study created in RDB with name: model_calibration_physical


Loaded study: model_calibration_physical
Objective version: cost_v2_physical
Stored trials: 0


## 5.2. Parameter distributions

Numeric bounds default to the observed minimum and maximum. Categorical choices come only from the explicit `ADVECTION_SCHEMES` configuration—not from numeric encoding or implicit ordering.

In [15]:
numeric_parameter_bounds = {
    parameter: (
        float(DF_optuna[parameter].min()),
        float(DF_optuna[parameter].max()),
    )
    for parameter in active_numeric_parameters
}

invalid_numeric_bounds = {
    parameter: bounds
    for parameter, bounds in numeric_parameter_bounds.items()
    if not np.isfinite(bounds).all() or bounds[0] >= bounds[1]
}
if invalid_numeric_bounds:
    raise ValueError(f"Invalid numeric parameter bounds: {invalid_numeric_bounds}")

categorical_choices = {
    CATEGORICAL_PARAMETER: tuple(ADVECTION_SCHEMES)
}

distribution_rows = [
    {"Parameter": parameter, "Type": "float", "Choices / Bounds": bounds}
    for parameter, bounds in numeric_parameter_bounds.items()
]
distribution_rows.extend(
    {"Parameter": parameter, "Type": "categorical", "Choices / Bounds": choices}
    for parameter, choices in categorical_choices.items()
)
display(pd.DataFrame(distribution_rows))

,Parameter,Type,Choices / Bounds
0,nl_tnu2_tracer0,float,"(0.5, 2.0)"
1,nl_tnu2_tracer1,float,"(0.5, 2.0)"
2,advection_scheme,categorical,"(Upstream3_Centered4, HSIMT, Akima4, MPDATA)"


## 5.3. Add historical runs

Historical runs teach Optuna about prior parameter–cost relationships. Numeric values are stored with float distributions and the advection bundle is stored with a categorical distribution. Runs with missing active values or unconfigured advection combinations are skipped and counted.

In [16]:
existing_run_names = {
    trial.user_attrs.get("Run Name")
    for trial in study.trials
    if trial.user_attrs.get("Run Name") is not None
}

import_counts = {
    "Added": 0,
    "Already present": 0,
    "Missing active parameters": 0,
    "Outside numeric bounds": 0,
    "Unrecognized category": 0,
}

float_distributions = {
    parameter: optuna.distributions.FloatDistribution(low=low, high=high)
    for parameter, (low, high) in numeric_parameter_bounds.items()
}
category_distributions = {
    parameter: optuna.distributions.CategoricalDistribution(choices=choices)
    for parameter, choices in categorical_choices.items()
}
trial_distributions = {**float_distributions, **category_distributions}

# Stable sorting makes fresh-study trial numbers follow model run dates.
# Runs assigned the same date retain their order from DF_optuna.
historical_trials = DF_optuna.sort_values(
    RUN_DATE_COLUMN,
    kind="stable",
)

for _, row in historical_trials.iterrows():
    run_name = row["Run Name"]
    if run_name in existing_run_names:
        import_counts["Already present"] += 1
        continue

    if row[active_parameters].isna().any():
        import_counts["Missing active parameters"] += 1
        continue

    if any(
        not (low <= float(row[parameter]) <= high)
        for parameter, (low, high) in numeric_parameter_bounds.items()
    ):
        import_counts["Outside numeric bounds"] += 1
        continue

    if any(
        row[parameter] not in choices
        for parameter, choices in categorical_choices.items()
    ):
        import_counts["Unrecognized category"] += 1
        continue

    params = {
        parameter: float(row[parameter])
        for parameter in active_numeric_parameters
    }
    params.update({
        parameter: str(row[parameter])
        for parameter in ACTIVE_CATEGORICAL_PARAMETERS
    })

    objective_values = {
        objective: (
            float(row[objective])
            if pd.notna(row[objective]) and np.isfinite(row[objective])
            else None
        )
        for objective in active_objective_cols
    }

    completed_trial = optuna.trial.create_trial(
        params=params,
        distributions=trial_distributions,
        value=float(row["Cost"]),
        state=optuna.trial.TrialState.COMPLETE,
        user_attrs={
            "Run Name": run_name,
            "Source": "Historical",
            "Objective Version": OBJECTIVE_VERSION,
            "Objectives": objective_values,
            RUN_DATE_COLUMN: row[RUN_DATE_COLUMN].date().isoformat(),
        },
    )
    study.add_trial(completed_trial)
    existing_run_names.add(run_name)
    import_counts["Added"] += 1

print("Historical trial import:")
for label, count in import_counts.items():
    print(f"  {label}: {count}")
print(f"Total Optuna trials: {len(study.trials)}")

Historical trial import:
  Added: 105
  Already present: 1
  Missing active parameters: 0
  Outside numeric bounds: 0
  Unrecognized category: 0
Total Optuna trials: 105


# 6. Candidates

## 6.1. Register newly completed model runs

A stored `RUNNING` trial is eligible for completion when its mapped actual run name appears in the cleaned table with a finite cost. If the trial is absent from the current study's run map, its existing run name is used. Mapped study run names and trial numbers must agree. Eligible trials are always displayed. The database changes only when `UPDATE_OPTUNA_WITH_COMPLETED_RUNS=True`.

In [17]:
duplicate_mask = DF_optuna["Run Name"].duplicated(keep=False)
duplicate_names = DF_optuna.loc[duplicate_mask, "Run Name"].unique()
for run_name in duplicate_names:
    duplicate_rows = DF_optuna.loc[DF_optuna["Run Name"].eq(run_name)]
    if len(duplicate_rows.drop_duplicates()) > 1:
        raise ValueError(f"Conflicting optimization rows for duplicate run {run_name!r}.")
if len(duplicate_names):
    DF_optuna = DF_optuna.drop_duplicates("Run Name", keep="first").reset_index(drop=True)

results_by_run = DF_optuna.set_index("Run Name", drop=False)
run_map_by_study_run_name = current_study_run_map.set_index(
    "Study Run Name", drop=False
)
run_map_by_trial_number = current_study_run_map.set_index(
    "Trial Number", drop=False
)
running_trials = [
    trial for trial in study.trials
    if trial.state == optuna.trial.TrialState.RUNNING
]
completion_rows = []
n_completed_now = 0
n_results_not_available = 0
n_missing_run_name = 0

for frozen_trial in running_trials:
    study_run_name = frozen_trial.user_attrs.get("Run Name")
    if not study_run_name:
        n_missing_run_name += 1
        print(f"Trial {frozen_trial.number}: missing Run Name attribute")
        continue

    if study_run_name in run_map_by_study_run_name.index:
        mapping = run_map_by_study_run_name.loc[study_run_name]
        mapped_trial_number = int(mapping["Trial Number"])
        if mapped_trial_number != frozen_trial.number:
            raise ValueError(
                f"Run-map mismatch for study {CURRENT_STUDY_NAME!r}: "
                f"Study Run Name {study_run_name!r} maps to trial "
                f"{mapped_trial_number}, but the RUNNING Optuna trial is "
                f"{frozen_trial.number}."
            )
        actual_run_name = mapping["Actual Run Name"]
    elif frozen_trial.number in run_map_by_trial_number.index:
        mapping = run_map_by_trial_number.loc[frozen_trial.number]
        raise ValueError(
            f"Run-map mismatch for study {CURRENT_STUDY_NAME!r}: trial "
            f"{frozen_trial.number} maps to Study Run Name "
            f"{mapping['Study Run Name']!r}, but the trial stores "
            f"{study_run_name!r}."
        )
    else:
        # The run map is an override table; unmapped trials keep existing behavior.
        actual_run_name = study_run_name

    if actual_run_name not in results_by_run.index:
        n_results_not_available += 1
        continue

    row = results_by_run.loc[actual_run_name]
    cost = float(row["Cost"])
    completion_rows.append({
        "Study Name": CURRENT_STUDY_NAME,
        "Actual Run Name": actual_run_name,
        "Study Run Name": study_run_name,
        "Trial Number": frozen_trial.number,
        RUN_DATE_COLUMN: row[RUN_DATE_COLUMN],
        "Cost": cost,
        "Action": "Complete in Optuna" if UPDATE_OPTUNA_WITH_COMPLETED_RUNS else "Preview only",
    })

    if UPDATE_OPTUNA_WITH_COMPLETED_RUNS:
        objective_values = {
            objective: float(row[objective])
            if pd.notna(row[objective]) and np.isfinite(row[objective]) else None
            for objective in active_objective_cols
        }
        live_trial = optuna.trial.Trial(study, frozen_trial._trial_id)
        live_trial.set_user_attr("Study Run Name", study_run_name)
        live_trial.set_user_attr("Actual Run Name", actual_run_name)
        live_trial.set_user_attr("Objectives", objective_values)
        live_trial.set_user_attr(
            RUN_DATE_COLUMN, row[RUN_DATE_COLUMN].date().isoformat()
        )
        study.tell(frozen_trial.number, cost)
        n_completed_now += 1

DF_completion_candidates = pd.DataFrame(
    completion_rows,
    columns=[
        "Study Name", "Actual Run Name", "Study Run Name",
        "Trial Number", RUN_DATE_COLUMN, "Cost", "Action",
    ],
)
if DF_completion_candidates.empty:
    print("No RUNNING trials have completed results available.")
else:
    display(DF_completion_candidates)

print("\nOptuna completion summary:")
print(f"  Update enabled:          {UPDATE_OPTUNA_WITH_COMPLETED_RUNS}")
print(f"  Eligible for completion: {len(completion_rows)}")
print(f"  Completed now:           {n_completed_now}")
print(f"  Results not available:   {n_results_not_available}")
print(f"  Missing Run Name:        {n_missing_run_name}")

No RUNNING trials have completed results available.

Optuna completion summary:
  Update enabled:          False
  Eligible for completion: 0
  Completed now:           0
  Results not available:   0
  Missing Run Name:        0


## 6.2. Generate the 12-run tied factorial grid

The controlled grid crosses three shared `nl_tnu2` values (`0.5`, `1.0`, and `2.0`) with the four configured advection bundles. Both `tracer0` and `tracer1` receive the same value, producing exactly $3 	imes 4 = 12$ combinations.

When `GENERATE_NEW_CANDIDATES=True`, the combinations are queued with fixed values and then claimed as `RUNNING` Optuna trials. This preserves the study-tracking workflow without allowing TPE to alter the factorial design. A grid-version label prevents the same experiment from being generated twice. The candidate CSV contains the bundle label and all four resolved advection fields, but no ROMS input files are created.

In [18]:
running_trials = [
    trial for trial in study.trials
    if trial.state == optuna.trial.TrialState.RUNNING
]
waiting_trials = [
    trial for trial in study.trials
    if trial.state == optuna.trial.TrialState.WAITING
]
existing_factorial_trials = [
    trial for trial in study.trials
    if trial.user_attrs.get("Factorial Grid Version") == FACTORIAL_GRID_VERSION
]

# Always expose current running trials so their candidate table can be recovered.
candidate_rows = []
for trial in running_trials:
    row = {
        "Run Name": trial.user_attrs.get("Run Name"),
        "Trial Number": trial.number,
        **trial.params,
    }
    scheme = row.get(CATEGORICAL_PARAMETER)
    if scheme in ADVECTION_SCHEMES:
        row.update(ADVECTION_SCHEMES[scheme])
    candidate_rows.append(row)

generated_new_candidates = False

if GENERATE_NEW_CANDIDATES and (running_trials or waiting_trials):
    print(
        "Not generating the factorial grid because the study already contains "
        f"{len(running_trials)} RUNNING and {len(waiting_trials)} WAITING trial(s)."
    )

elif GENERATE_NEW_CANDIDATES and existing_factorial_trials:
    print(
        f"Grid {FACTORIAL_GRID_VERSION!r} already exists with "
        f"{len(existing_factorial_trials)} trial(s); it will not be generated again."
    )

elif GENERATE_NEW_CANDIDATES:
    grid_specs = []

    # Build every advection x shared-nl_tnu2 combination exactly once.
    for scheme in GRID_ADVECTION_SCHEMES:
        for nl_tnu2_value in NL_TNU2_GRID:
            grid_specs.append({
                "Grid Index": len(grid_specs) + 1,
                "Run Name": f"PHYSICAL_GRID_{len(grid_specs) + 1:03d}",
                "Parameters": {
                    "nl_tnu2_tracer0": float(nl_tnu2_value),
                    "nl_tnu2_tracer1": float(nl_tnu2_value),
                    CATEGORICAL_PARAMETER: scheme,
                },
            })

    if len(grid_specs) != 12:
        raise RuntimeError(f"Expected 12 factorial combinations; found {len(grid_specs)}.")

    # Queue fixed parameter dictionaries so study.ask() cannot substitute TPE values.
    for spec in grid_specs:
        study.enqueue_trial(
            spec["Parameters"],
            user_attrs={
                "Run Name": spec["Run Name"],
                "Source": "Controlled factorial grid",
                "Objective Version": OBJECTIVE_VERSION,
                "Factorial Grid Version": FACTORIAL_GRID_VERSION,
                "Grid Index": spec["Grid Index"],
            },
        )

    # Claim every queued combination as a RUNNING trial and register distributions.
    for expected_spec in grid_specs:
        trial = study.ask()
        params = {
            parameter: trial.suggest_float(parameter, low, high)
            for parameter, (low, high) in numeric_parameter_bounds.items()
        }
        params[CATEGORICAL_PARAMETER] = trial.suggest_categorical(
            CATEGORICAL_PARAMETER,
            categorical_choices[CATEGORICAL_PARAMETER],
        )

        if params != expected_spec["Parameters"]:
            raise RuntimeError(
                "Claimed Optuna trial did not match the queued factorial combination: "
                f"expected {expected_spec['Parameters']}, received {params}."
            )

        candidate_rows.append({
            "Run Name": trial.user_attrs["Run Name"],
            "Trial Number": trial.number,
            **params,
            **ADVECTION_SCHEMES[params[CATEGORICAL_PARAMETER]],
        })

    generated_new_candidates = True
    print(f"Generated all {len(grid_specs)} combinations in {FACTORIAL_GRID_VERSION!r}.")

else:
    print("Factorial candidate generation is OFF.")

candidate_columns = [
    "Run Name", "Trial Number", *active_numeric_parameters,
    CATEGORICAL_PARAMETER, *ADVECTION_COLUMNS,
]
DF_candidates = pd.DataFrame(candidate_rows).reindex(columns=candidate_columns)
display(DF_candidates)

Generated all 12 combinations in 'physical_tied_grid_v1'.


,Run Name,Trial Number,nl_tnu2_tracer0,nl_tnu2_tracer1,advection_scheme,temp_horizontal_advection,temp_vertical_advection,salt_horizontal_advection,salt_vertical_advection
0,PHYSICAL_GRID_001,105,0.5,0.5,Upstream3_Centered4,Upstream3,Centered4,Upstream3,Centered4
1,PHYSICAL_GRID_002,106,1.0,1.0,Upstream3_Centered4,Upstream3,Centered4,Upstream3,Centered4
2,PHYSICAL_GRID_003,107,2.0,2.0,Upstream3_Centered4,Upstream3,Centered4,Upstream3,Centered4
3,PHYSICAL_GRID_004,108,0.5,0.5,HSIMT,HSIMT,HSIMT,HSIMT,HSIMT
4,PHYSICAL_GRID_005,109,1.0,1.0,HSIMT,HSIMT,HSIMT,HSIMT,HSIMT
5,PHYSICAL_GRID_006,110,2.0,2.0,HSIMT,HSIMT,HSIMT,HSIMT,HSIMT
6,PHYSICAL_GRID_007,111,0.5,0.5,Akima4,Akima4,Akima4,Akima4,Akima4
7,PHYSICAL_GRID_008,112,1.0,1.0,Akima4,Akima4,Akima4,Akima4,Akima4
8,PHYSICAL_GRID_009,113,2.0,2.0,Akima4,Akima4,Akima4,Akima4,Akima4
9,PHYSICAL_GRID_010,114,0.5,0.5,MPDATA,MPDATA,MPDATA,MPDATA,MPDATA


In [19]:
# Save only a newly requested batch; viewing existing RUNNING trials does not rewrite the CSV.
if generated_new_candidates:
    candidate_file = OPT_DIR / "Optuna_Physical_Candidate_Parameters.csv"
    DF_candidates.to_csv(candidate_file, index=False)
    print(f"Saved candidate parameters to: {candidate_file}")
else:
    print("No new candidate CSV was written.")

Saved candidate parameters to: /Users/akbaskind/Desktop/Optimization/Optuna_Physical_Candidate_Parameters.csv


## Stopping point

The workflow intentionally ends here. When generation is enabled, `DF_candidates` contains the complete 12-run factorial design: three tied temperature/salinity `nl_tnu2` values crossed with four advection bundles. Translating these values into ROMS configuration files is deferred until the physical input template and replacement rules are decided.

In [20]:
# ============================================================
# FINAL STUDY STATUS
# ============================================================

print(f"Study: {study.study_name}")
print(f"Total trials: {len(study.trials)}")

for state in optuna.trial.TrialState:
    count = sum(trial.state == state for trial in study.trials)
    if count > 0:
        print(f"  {state.name}: {count}")

Study: model_calibration_physical
Total trials: 117
  RUNNING: 12
  COMPLETE: 105


In [25]:
dstest = xr.open_dataset("/Users/akbaskind/Desktop/COST_FILES/ocean_sta_Ca1.nc")
dstest

<xarray.Dataset> Size: 5GB
Dimensions:             (tracer: 18, boundary: 4, nspc: 3, s_rho: 15, s_w: 16,
                         station: 188, ocean_time: 17521)
Coordinates:
  * s_rho               (s_rho) float64 120B -0.9667 -0.9 ... -0.1 -0.03333
  * s_w                 (s_w) float64 128B -1.0 -0.9333 -0.8667 ... -0.06667 0.0
    lon_rho             (station) float64 2kB ...
    lat_rho             (station) float64 2kB ...
  * ocean_time          (ocean_time) datetime64[ns] 140kB 2005-01-01 ... 2006...
Dimensions without coordinates: tracer, boundary, nspc, station
Data variables: (12/224)
    ntimes              int32 4B ...
    ndtfast             int32 4B ...
    dt                  float64 8B ...
    dtfast              float64 8B ...
    dstart              datetime64[ns] 8B ...
    shuffle             int32 4B ...
    ...                  ...
    benthic_flux_NH4    (ocean_time, station) float32 13MB ...
    benthic_flux_PO4    (ocean_time, station) float32 13MB ...
    benthic_flux_Si     (ocean_time, station) float32 13MB ...
    SOD                 (ocean_time, station) float32 13MB ...
    benthic_flux_TIC    (ocean_time, station) float32 13MB ...
    benthic_flux_Alk    (ocean_time, station) float32 13MB ...
Attributes: (12/38)
    file:              /project/pi_dullman_uri_edu/abby/CaCO3/ocean_sta_Ca1.nc
    format:            netCDF-4/HDF5 file
    Conventions:       CF-1.4, SGRID-0.3
    type:              ROMS station file
    title:             OSOM_COSINE
    var_info:          /home/abaskind_uri_edu/roms4.2_cosine/ROMS/External/va...
    ...                ...
    compiler_flags:    -frepack-arrays -fallow-argument-mismatch        -O3 -...
    tiling:            16x16
    history:           ROMS, Version 4.3, Sunday - June 7, 2026 - 10:08:53 AM
    ana_file:          ROMS/Functionals/ana_btflux.h, ROMS/Functionals/ana_st...
    CPP_options:       EPSCOR_UMAINE15_SEDBIO_OPTICS_ZENITH_DIAG_PHYRES_V4.2_...
    bio_file:          ROMS/Nonlinear/Biology/bio_UMAINE15.h

In [26]:
dstest.attrs.get("CPP_options", "Attribute not found")

'EPSCOR_UMAINE15_SEDBIO_OPTICS_ZENITH_DIAG_PHYRES_V4.2_SHORT, ADD_FSOBC, ADD_M2OBC, ANA_BPFLUX, ANA_BSFLUX, ANA_BTFLUX, ANA_SPFLUX, ASSUMED_SHAPE, BIO_UMAINE15, SEDBIO, OPTICS_OP1, READ_ZENITH, CARBON, TALK_NONCONSERV, OXYGEN, PHYTO_RESP, PCO2AIR_SEA CACO3, BOUNDARY_ALLREDUCE, BULK_FLUXES, CANUTO_A COLLECT_ALLGATHER, CURVGRID, DEFLATE, DIFF_GRID, DJ_GRADPS, DOUBLE_PRECISION, EMINUSP, !GATHER_SENDRECV, GLS_MIXING, HDF5, LONGWAVE_OUT, MASKING, MIX_ISO_TS, MIX_GEO_UV, MPI, !MULTIPLE_THREAD, NONLINEAR, NONLIN_EOS, NO_WRITE_GRID, N2S2_HORAVG, OMEGA_IMPLICIT, PERFECT_RESTART, POWER_LAW, PROFILE, K_GSCHEME, RADIATION_2D, RAMP_TIDES, REDUCE_ALLREDUCE, !RST_SINGLE, SALINITY, !SCATTER_BCAST, STEP2D_LF_AM3, SOLAR_SOURCE, SOLVE3D, SSH_TIDES, STATIONS, TS_DIF2, UV_ADV, UV_COR, UV_U3HADVECTION, UV_C4VADVECTION, UV_LOGDRAG, UV_TIDES, UV_VIS2, VAR_RHO_2D, VISC_GRID'

In [ ]:
cacopf, cacodr, omega_thresh, wsPCa, bUmaxCa

i need to add more information from the station file to my optuna database. this will apply to CandidateParameterSetsBio.ipynb. do not make any changes until we review the issue and the proposed changes.

the first thing I hope to happen is check whether the "CACO3" option was activated in a given run. This can be found in the station under the attribute "CPP_options." That attribute will give you a list of the CPP options used. If CACO3 is in that list, then CACO3 was activated. i would like whether CACO3 was on to be represented as a categorical parameter. 

An example of this can be found in "/Users/akbaskind/Desktop/COST_FILES/ocean_sta_Ca1.nc" where the list of CPP options looks like this: 'EPSCOR_UMAINE15_SEDBIO_OPTICS_ZENITH_DIAG_PHYRES_V4.2_SHORT, ADD_FSOBC, ADD_M2OBC, ANA_BPFLUX, ANA_BSFLUX, ANA_BTFLUX, ANA_SPFLUX, ASSUMED_SHAPE, BIO_UMAINE15, SEDBIO, OPTICS_OP1, READ_ZENITH, CARBON, TALK_NONCONSERV, OXYGEN, PHYTO_RESP, PCO2AIR_SEA CACO3, BOUNDARY_ALLREDUCE, BULK_FLUXES, CANUTO_A COLLECT_ALLGATHER, CURVGRID, DEFLATE, DIFF_GRID, DJ_GRADPS, DOUBLE_PRECISION, EMINUSP, !GATHER_SENDRECV, GLS_MIXING, HDF5, LONGWAVE_OUT, MASKING, MIX_ISO_TS, MIX_GEO_UV, MPI, !MULTIPLE_THREAD, NONLINEAR, NONLIN_EOS, NO_WRITE_GRID, N2S2_HORAVG, OMEGA_IMPLICIT, PERFECT_RESTART, POWER_LAW, PROFILE, K_GSCHEME, RADIATION_2D, RAMP_TIDES, REDUCE_ALLREDUCE, !RST_SINGLE, SALINITY, !SCATTER_BCAST, STEP2D_LF_AM3, SOLAR_SOURCE, SOLVE3D, SSH_TIDES, STATIONS, TS_DIF2, UV_ADV, UV_COR, UV_U3HADVECTION, UV_C4VADVECTION, UV_LOGDRAG, UV_TIDES, UV_VIS2, VAR_RHO_2D, VISC_GRID'

If CACO3 is on based on that attribute, i would like to identify the values of the following parameters: cacopf, cacodr, omega_thresh, wsPCa, bUmaxCa. These parameters can be added as float parameters. These parameters can be found in the station file data variables.

Note that runs that do not have CACO3 on will not necessarily have those numerical parameters defined. Rather than search for them when CACO3 is off, they should automatically be set to 0.0.

None of the runs incldued in FileList.xlsx currently have CACO3 turned on.

I would like this to be added to the information gleaned from the file reads in CandidateParameterSetsBio.ipynb and I would like it to be nicely commented and documented. Think you can manage that? Before making any changes, please confirm your understanding please and thank you.